<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/08_Confidence_%2B_Abstention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 08.0 — ENVIRONMENT & CONFIGURATION
# ============================================================

from pathlib import Path
import json
import hashlib
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08")
print("CONFIDENCE ESTIMATION + ABSTENTION")
print("=" * 100)


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

from google.colab import drive

MOUNT_POINT = Path("/content/drive")

try:
    if not (MOUNT_POINT / "MyDrive").exists():
        drive.mount(str(MOUNT_POINT), force_remount=False)
except Exception:
    drive.mount(str(MOUNT_POINT), force_remount=True)


# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = (
    MOUNT_POINT
    / "MyDrive"
    / "AIR_LLM_Research"
)

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"AIR-LLM project not found:\n{PROJECT_ROOT}"
    )


# ------------------------------------------------------------
# Notebook directories
# ------------------------------------------------------------

NB03_DIR = PROJECT_ROOT / "data" / "notebook_03"
NB04_DIR = PROJECT_ROOT / "data" / "notebook_04"
NB05_DIR = PROJECT_ROOT / "data" / "notebook_05"
NB07_DIR = PROJECT_ROOT / "data" / "notebook_07"

NB08_DIR = PROJECT_ROOT / "data" / "notebook_08"

PROFILE_DIR = NB04_DIR / "profiles"
RECOMMENDATION_DIR = NB05_DIR / "recommendations"
UTILITY_DIR = NB07_DIR / "utility"

CONFIDENCE_DIR = NB08_DIR / "confidence"
ABSTENTION_DIR = NB08_DIR / "abstention"
STABILITY_DIR = NB08_DIR / "stability"
SUMMARY_DIR = NB08_DIR / "summary"
METADATA_DIR = NB08_DIR / "metadata"

for directory in [
    NB08_DIR,
    CONFIDENCE_DIR,
    ABSTENTION_DIR,
    STABILITY_DIR,
    SUMMARY_DIR,
    METADATA_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Experimental configuration
# ------------------------------------------------------------

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR_APPROXIMATION"
]

MISSINGNESS_RATES = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

REPETITIONS = 5

MASTER_SEED = 42

# ------------------------------------------------------------
# Confidence threshold
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.60

# Minimum utility margin considered meaningful
UTILITY_MARGIN_THRESHOLD = 0.05

# Stability tolerance
STABILITY_TOLERANCE = 0.05


print("\nPROJECT")
print("-" * 100)
print(f"Project root        : {PROJECT_ROOT}")
print(f"Notebook 08 output  : {NB08_DIR}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"Utility margin      : {UTILITY_MARGIN_THRESHOLD}")
print(f"Stability tolerance : {STABILITY_TOLERANCE}")

print("\nNOTEBOOK 08 CONFIGURATION READY")

In [ ]:
# ============================================================
# NOTEBOOK 08.1 — ARTIFACT DISCOVERY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.1")
print("DISCOVERING PREVIOUS NOTEBOOK ARTIFACTS")
print("=" * 100)


def find_csv_candidates(root):
    if not root.exists():
        return []

    return sorted(
        root.rglob("*.csv")
    )


UTILITY_FILES = find_csv_candidates(
    UTILITY_DIR
)

RECOMMENDATION_FILES = find_csv_candidates(
    RECOMMENDATION_DIR
)

PROFILE_FILES = find_csv_candidates(
    PROFILE_DIR
)


print("\nUTILITY FILES")
print("-" * 100)

for path in UTILITY_FILES:
    print(path)


print("\nRECOMMENDATION FILES")
print("-" * 100)

for path in RECOMMENDATION_FILES:
    print(path)


print("\nPROFILE FILES")
print("-" * 100)

for path in PROFILE_FILES:
    print(path)

In [ ]:
# ============================================================
# NOTEBOOK 08.2 — LOAD ADAPTIVE UTILITY RESULTS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.2")
print("LOADING ADAPTIVE UTILITY RESULTS")
print("=" * 100)


def choose_file(files, keywords):

    scored = []

    for path in files:

        name = path.name.lower()

        score = sum(
            1
            for keyword in keywords
            if keyword.lower() in name
        )

        scored.append(
            (score, path)
        )

    scored.sort(
        key=lambda x: (-x[0], str(x[1]))
    )

    if scored and scored[0][0] > 0:
        return scored[0][1]

    return None


UTILITY_PATH = choose_file(
    UTILITY_FILES,
    [
        "utility",
        "selection",
        "adaptive",
        "result"
    ]
)


if UTILITY_PATH is None:

    raise FileNotFoundError(
        "No adaptive utility result CSV was found in:\n"
        f"{UTILITY_DIR}"
    )


UTILITY_DF = pd.read_csv(
    UTILITY_PATH
)


print("\nUTILITY FILE")
print("-" * 100)
print(UTILITY_PATH)

print("\nSHAPE")
print("-" * 100)
print(UTILITY_DF.shape)

print("\nCOLUMNS")
print("-" * 100)

for column in UTILITY_DF.columns:
    print(column)

display(
    UTILITY_DF.head()
)

In [ ]:
# ============================================================
# NOTEBOOK 08.3 — NORMALIZE UTILITY SCHEMA
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.3")
print("NORMALIZING UTILITY RESULT SCHEMA")
print("=" * 100)


def find_column(df, candidates, required=True):

    lower_map = {
        c.lower(): c
        for c in df.columns
    }

    # Exact match
    for candidate in candidates:

        if candidate.lower() in lower_map:
            return lower_map[
                candidate.lower()
            ]

    # Partial match
    for candidate in candidates:

        for column in df.columns:

            if candidate.lower() in column.lower():
                return column

    if required:
        raise KeyError(
            f"Required column not found. "
            f"Candidates: {candidates}\n"
            f"Available: {list(df.columns)}"
        )

    return None


UTILITY_SCHEMA = {}


UTILITY_SCHEMA["dataset_id"] = find_column(
    UTILITY_DF,
    ["dataset_id", "dataset"]
)

UTILITY_SCHEMA["feature"] = find_column(
    UTILITY_DF,
    ["feature", "feature_name", "column"]
)

UTILITY_SCHEMA["strategy"] = find_column(
    UTILITY_DF,
    [
        "strategy",
        "method",
        "candidate_strategy",
        "selected_strategy"
    ]
)

UTILITY_SCHEMA["utility"] = find_column(
    UTILITY_DF,
    [
        "utility",
        "composite_utility",
        "normalized_utility",
        "final_utility"
    ]
)

UTILITY_SCHEMA["rank"] = find_column(
    UTILITY_DF,
    [
        "rank",
        "utility_rank",
        "selection_rank"
    ],
    required=False
)


print("\nUTILITY SCHEMA")
print("-" * 100)

for key, value in UTILITY_SCHEMA.items():
    print(f"{key:15s} : {value}")


UTILITY = UTILITY_DF.copy()

UTILITY["dataset_id"] = (
    UTILITY[
        UTILITY_SCHEMA["dataset_id"]
    ].astype(str)
)

UTILITY["feature"] = (
    UTILITY[
        UTILITY_SCHEMA["feature"]
    ].astype(str)
)

UTILITY["strategy"] = (
    UTILITY[
        UTILITY_SCHEMA["strategy"]
    ].astype(str)
)

UTILITY["utility"] = pd.to_numeric(
    UTILITY[
        UTILITY_SCHEMA["utility"]
    ],
    errors="coerce"
)

UTILITY = UTILITY.dropna(
    subset=[
        "dataset_id",
        "feature",
        "strategy",
        "utility"
    ]
)

print(
    f"\nValid utility rows: {len(UTILITY):,}"
)

In [ ]:
# ============================================================
# NOTEBOOK 08.4 — WINNING STRATEGY IDENTIFICATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.4")
print("IDENTIFYING TOP STRATEGIES")
print("=" * 100)


UTILITY_SORTED = UTILITY.sort_values(
    [
        "dataset_id",
        "feature",
        "utility"
    ],
    ascending=[
        True,
        True,
        False
    ]
).copy()


UTILITY_SORTED["utility_rank_computed"] = (
    UTILITY_SORTED
    .groupby(
        [
            "dataset_id",
            "feature"
        ]
    )["utility"]
    .rank(
        method="first",
        ascending=False
    )
)


TOP_STRATEGIES = (
    UTILITY_SORTED[
        UTILITY_SORTED[
            "utility_rank_computed"
        ] == 1
    ]
    .copy()
)


TOP_STRATEGIES = TOP_STRATEGIES[
    [
        "dataset_id",
        "feature",
        "strategy",
        "utility"
    ]
].rename(
    columns={
        "strategy":
            "selected_strategy",

        "utility":
            "selected_utility"
    }
)


print(
    f"Feature-level decisions: "
    f"{len(TOP_STRATEGIES):,}"
)

display(
    TOP_STRATEGIES.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 08.5 — EMPIRICAL PERFORMANCE MARGIN
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.5")
print("EMPIRICAL PERFORMANCE MARGIN")
print("=" * 100)


GROUP_KEYS = [
    "dataset_id",
    "feature"
]


UTILITY_MARGIN = (
    UTILITY_SORTED
    .groupby(GROUP_KEYS)["utility"]
    .apply(
        lambda x:
        float(x.iloc[0])
        -
        float(x.iloc[1])
        if len(x) >= 2
        else 0.0
    )
    .reset_index(
        name="utility_margin"
    )
)


UTILITY_MARGIN["margin_confidence"] = (
    UTILITY_MARGIN[
        "utility_margin"
    ]
    .clip(
        lower=0.0,
        upper=1.0
    )
    /
    max(
        UTILITY_MARGIN_THRESHOLD,
        1e-12
    )
).clip(
    upper=1.0
)


display(
    UTILITY_MARGIN.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 08.5 — EMPIRICAL PERFORMANCE MARGIN
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.5")
print("EMPIRICAL PERFORMANCE MARGIN")
print("=" * 100)


GROUP_KEYS = [
    "dataset_id",
    "feature"
]


UTILITY_MARGIN = (
    UTILITY_SORTED
    .groupby(GROUP_KEYS)["utility"]
    .apply(
        lambda x:
        float(x.iloc[0])
        -
        float(x.iloc[1])
        if len(x) >= 2
        else 0.0
    )
    .reset_index(
        name="utility_margin"
    )
)


UTILITY_MARGIN["margin_confidence"] = (
    UTILITY_MARGIN[
        "utility_margin"
    ]
    .clip(
        lower=0.0,
        upper=1.0
    )
    /
    max(
        UTILITY_MARGIN_THRESHOLD,
        1e-12
    )
).clip(
    upper=1.0
)


display(
    UTILITY_MARGIN.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 08.7 — LLM RECOMMENDATION CONFIDENCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.7")
print("LLM RECOMMENDATION CONFIDENCE")
print("=" * 100)


LLM_CONFIDENCE = None


if RECOMMENDATION_FILES:

    candidate = choose_file(
        RECOMMENDATION_FILES,
        [
            "recommendation",
            "recommendations",
            "ranked"
        ]
    )

    if candidate is not None:

        try:

            REC_DF = pd.read_csv(
                candidate
            )

            print(
                f"Recommendation file: {candidate}"
            )

            print(
                f"Rows: {len(REC_DF):,}"
            )

            confidence_column = find_column(
                REC_DF,
                [
                    "confidence",
                    "llm_confidence",
                    "recommendation_confidence"
                ],
                required=False
            )

            dataset_column = find_column(
                REC_DF,
                [
                    "dataset_id",
                    "dataset"
                ],
                required=False
            )

            feature_column = find_column(
                REC_DF,
                [
                    "feature",
                    "feature_name"
                ],
                required=False
            )

            strategy_column = find_column(
                REC_DF,
                [
                    "strategy",
                    "candidate_strategy",
                    "method"
                ],
                required=False
            )

            if (
                confidence_column
                and dataset_column
                and feature_column
                and strategy_column
            ):

                LLM_CONFIDENCE = REC_DF.copy()

                LLM_CONFIDENCE[
                    "dataset_id"
                ] = (
                    LLM_CONFIDENCE[
                        dataset_column
                    ].astype(str)
                )

                LLM_CONFIDENCE[
                    "feature"
                ] = (
                    LLM_CONFIDENCE[
                        feature_column
                    ].astype(str)
                )

                LLM_CONFIDENCE[
                    "strategy"
                ] = (
                    LLM_CONFIDENCE[
                        strategy_column
                    ].astype(str)
                )

                LLM_CONFIDENCE[
                    "llm_confidence"
                ] = pd.to_numeric(
                    LLM_CONFIDENCE[
                        confidence_column
                    ],
                    errors="coerce"
                )

                LLM_CONFIDENCE[
                    "llm_confidence"
                ] = (
                    LLM_CONFIDENCE[
                        "llm_confidence"
                    ]
                    .clip(
                        0.0,
                        1.0
                    )
                )

                LLM_CONFIDENCE = (
                    LLM_CONFIDENCE[
                        [
                            "dataset_id",
                            "feature",
                            "strategy",
                            "llm_confidence"
                        ]
                    ]
                )

            else:

                print(
                    "No compatible LLM confidence column found."
                )

        except Exception as exc:

            print(
                "Recommendation confidence could not be loaded."
            )

            print(
                f"Reason: {exc}"
            )


if LLM_CONFIDENCE is None:

    print(
        "\nUsing neutral auxiliary LLM confidence = 0.5."
    )

    LLM_CONFIDENCE = pd.DataFrame({
        "dataset_id":
            TOP_STRATEGIES[
                "dataset_id"
            ],

        "feature":
            TOP_STRATEGIES[
                "feature"
            ],

        "strategy":
            TOP_STRATEGIES[
                "selected_strategy"
            ],

        "llm_confidence":
            0.5
    })


display(
    LLM_CONFIDENCE.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 08.8 — CONFIDENCE COMPONENT ASSEMBLY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.8")
print("ASSEMBLING CONFIDENCE COMPONENTS")
print("=" * 100)


CONFIDENCE_DF = TOP_STRATEGIES.copy()


CONFIDENCE_DF = CONFIDENCE_DF.merge(
    UTILITY_MARGIN,
    on=[
        "dataset_id",
        "feature"
    ],
    how="left"
)


CONFIDENCE_DF = CONFIDENCE_DF.merge(
    STABILITY_DF[
        [
            "dataset_id",
            "feature",
            "stability_score"
        ]
    ],
    on=[
        "dataset_id",
        "feature"
    ],
    how="left"
)


# ------------------------------------------------------------
# Merge LLM confidence
# ------------------------------------------------------------

LLM_SELECTED = (
    LLM_CONFIDENCE
    .sort_values(
        "llm_confidence",
        ascending=False
    )
    .drop_duplicates(
        [
            "dataset_id",
            "feature",
            "strategy"
        ]
    )
)


CONFIDENCE_DF = CONFIDENCE_DF.merge(
    LLM_SELECTED,
    left_on=[
        "dataset_id",
        "feature",
        "selected_strategy"
    ],
    right_on=[
        "dataset_id",
        "feature",
        "strategy"
    ],
    how="left"
)


if "strategy" in CONFIDENCE_DF.columns:
    CONFIDENCE_DF = CONFIDENCE_DF.drop(
        columns=["strategy"]
    )


CONFIDENCE_DF[
    "llm_confidence"
] = (
    CONFIDENCE_DF[
        "llm_confidence"
    ]
    .fillna(0.5)
    .clip(0.0, 1.0)
)


CONFIDENCE_DF[
    "stability_score"
] = (
    CONFIDENCE_DF[
        "stability_score"
    ]
    .fillna(0.0)
    .clip(0.0, 1.0)
)


CONFIDENCE_DF[
    "margin_confidence"
] = (
    CONFIDENCE_DF[
        "margin_confidence"
    ]
    .fillna(0.0)
    .clip(0.0, 1.0)
)


print(
    f"Confidence rows: "
    f"{len(CONFIDENCE_DF):,}"
)


display(
    CONFIDENCE_DF.head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 08.9 — FINAL CONFIDENCE SCORE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.9")
print("FINAL CONFIDENCE ESTIMATION")
print("=" * 100)


# ------------------------------------------------------------
# Confidence weights
# ------------------------------------------------------------

W_LLM = 0.20
W_MARGIN = 0.40
W_STABILITY = 0.40


CONFIDENCE_DF[
    "final_confidence"
] = (

    W_LLM
    *
    CONFIDENCE_DF[
        "llm_confidence"
    ]

    +

    W_MARGIN
    *
    CONFIDENCE_DF[
        "margin_confidence"
    ]

    +

    W_STABILITY
    *
    CONFIDENCE_DF[
        "stability_score"
    ]
)


CONFIDENCE_DF[
    "final_confidence"
] = (
    CONFIDENCE_DF[
        "final_confidence"
    ]
    .clip(
        0.0,
        1.0
    )
)


CONFIDENCE_DF[
    "confidence_level"
] = pd.cut(
    CONFIDENCE_DF[
        "final_confidence"
    ],
    bins=[
        -np.inf,
        0.40,
        0.60,
        0.80,
        np.inf
    ],
    labels=[
        "LOW",
        "MODERATE",
        "HIGH",
        "VERY_HIGH"
    ]
)


display(
    CONFIDENCE_DF[
        [
            "dataset_id",
            "feature",
            "selected_strategy",
            "llm_confidence",
            "margin_confidence",
            "stability_score",
            "final_confidence",
            "confidence_level"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# NOTEBOOK 08.10 — ABSTENTION MECHANISM
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.10")
print("ABSTENTION MECHANISM")
print("=" * 100)


CONFIDENCE_DF[
    "abstain"
] = (
    CONFIDENCE_DF[
        "final_confidence"
    ]
    <
    CONFIDENCE_THRESHOLD
)


CONFIDENCE_DF[
    "decision"
] = np.where(
    CONFIDENCE_DF[
        "abstain"
    ],
    "ABSTAIN",
    "ACCEPT"
)


CONFIDENCE_DF[
    "decision_reason"
] = np.where(

    CONFIDENCE_DF[
        "abstain"
    ],

    "Confidence below predefined threshold; additional candidate evaluation required.",

    "Confidence sufficient for adaptive strategy acceptance."
)


print("\nDECISION COUNTS")
print("-" * 100)

print(
    CONFIDENCE_DF[
        "decision"
    ]
    .value_counts()
)


display(
    CONFIDENCE_DF[
        [
            "dataset_id",
            "feature",
            "selected_strategy",
            "final_confidence",
            "confidence_level",
            "decision",
            "decision_reason"
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# NOTEBOOK 08.11 — ABSTENTION RECOVERY CANDIDATES
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.11")
print("ABSTENTION RECOVERY CANDIDATES")
print("=" * 100)


ABSTAINED_KEYS = CONFIDENCE_DF[
    CONFIDENCE_DF[
        "abstain"
    ]
][
    [
        "dataset_id",
        "feature"
    ]
].drop_duplicates()


if len(ABSTAINED_KEYS) > 0:

    RECOVERY = UTILITY_SORTED.merge(
        ABSTAINED_KEYS,
        on=[
            "dataset_id",
            "feature"
        ],
        how="inner"
    )

    RECOVERY[
        "candidate_rank"
    ] = (
        RECOVERY
        .groupby(
            [
                "dataset_id",
                "feature"
            ]
        )["utility"]
        .rank(
            method="first",
            ascending=False
        )
    )

    RECOVERY = RECOVERY[
        RECOVERY[
            "candidate_rank"
        ] <= 5
    ].copy()

    RECOVERY_PATH = (
        ABSTENTION_DIR
        / "abstention_recovery_candidates.csv"
    )

    RECOVERY.to_csv(
        RECOVERY_PATH,
        index=False
    )

    display(
        RECOVERY[
            [
                "dataset_id",
                "feature",
                "strategy",
                "utility",
                "candidate_rank"
            ]
        ].head(30)
    )

else:

    RECOVERY = pd.DataFrame()

    RECOVERY_PATH = (
        ABSTENTION_DIR
        / "abstention_recovery_candidates.csv"
    )

    RECOVERY.to_csv(
        RECOVERY_PATH,
        index=False
    )

    print(
        "No features require abstention recovery."
    )

In [ ]:
# ============================================================
# NOTEBOOK 08.12 — CONFIDENCE STABILITY SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.12")
print("CONFIDENCE STABILITY SUMMARY")
print("=" * 100)


STABILITY_SUMMARY = (
    CONFIDENCE_DF
    .groupby(
        "dataset_id"
    )
    .agg(
        features=(
            "feature",
            "nunique"
        ),

        mean_confidence=(
            "final_confidence",
            "mean"
        ),

        median_confidence=(
            "final_confidence",
            "median"
        ),

        min_confidence=(
            "final_confidence",
            "min"
        ),

        accepted_features=(
            "abstain",
            lambda x:
            int((~x).sum())
        ),

        abstained_features=(
            "abstain",
            "sum"
        )
    )
    .reset_index()
)


STABILITY_SUMMARY[
    "abstention_rate"
] = (
    STABILITY_SUMMARY[
        "abstained_features"
    ]
    /
    STABILITY_SUMMARY[
        "features"
    ]
)


display(
    STABILITY_SUMMARY
)

In [ ]:
# ============================================================
# NOTEBOOK 08.13 — CONFIDENCE PERSISTENCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.13")
print("SAVING CONFIDENCE AND ABSTENTION RESULTS")
print("=" * 100)


CONFIDENCE_PATH = (
    CONFIDENCE_DIR
    / "feature_confidence.csv"
)

DECISION_PATH = (
    ABSTENTION_DIR
    / "abstention_decisions.csv"
)

STABILITY_PATH = (
    STABILITY_DIR
    / "confidence_stability.csv"
)

SUMMARY_PATH = (
    SUMMARY_DIR
    / "confidence_summary.csv"
)


CONFIDENCE_DF.to_csv(
    CONFIDENCE_PATH,
    index=False
)


CONFIDENCE_DF[
    [
        "dataset_id",
        "feature",
        "selected_strategy",
        "selected_utility",
        "final_confidence",
        "confidence_level",
        "abstain",
        "decision",
        "decision_reason"
    ]
].to_csv(
    DECISION_PATH,
    index=False
)


STABILITY_DF.to_csv(
    STABILITY_PATH,
    index=False
)


STABILITY_SUMMARY.to_csv(
    SUMMARY_PATH,
    index=False
)


print("\nSAVED")
print("-" * 100)

print(
    f"Confidence : {CONFIDENCE_PATH}"
)

print(
    f"Decisions  : {DECISION_PATH}"
)

print(
    f"Stability  : {STABILITY_PATH}"
)

print(
    f"Summary    : {SUMMARY_PATH}"
)

In [ ]:
# ============================================================
# NOTEBOOK 08.14 — FINAL VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.14")
print("FINAL CONFIDENCE AND ABSTENTION VALIDATION")
print("=" * 100)


VALIDATION = {}


# ------------------------------------------------------------
# Row coverage
# ------------------------------------------------------------

VALIDATION[
    "confidence_rows"
] = (
    len(CONFIDENCE_DF)
    > 0
)


# ------------------------------------------------------------
# Confidence range
# ------------------------------------------------------------

VALIDATION[
    "confidence_range"
] = bool(
    (
        CONFIDENCE_DF[
            "final_confidence"
        ].between(
            0.0,
            1.0
        )
    ).all()
)


# ------------------------------------------------------------
# Decision consistency
# ------------------------------------------------------------

VALIDATION[
    "decision_consistency"
] = bool(
    (
        (
            CONFIDENCE_DF[
                "final_confidence"
            ]
            <
            CONFIDENCE_THRESHOLD
        )
        ==
        CONFIDENCE_DF[
            "abstain"
        ]
    ).all()
)


# ------------------------------------------------------------
# No missing selected strategies
# ------------------------------------------------------------

VALIDATION[
    "selected_strategy_present"
] = bool(
    CONFIDENCE_DF[
        "selected_strategy"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# Persistence
# ------------------------------------------------------------

VALIDATION[
    "confidence_persisted"
] = CONFIDENCE_PATH.is_file()


VALIDATION[
    "decision_persisted"
] = DECISION_PATH.is_file()


VALIDATION[
    "stability_persisted"
] = STABILITY_PATH.is_file()


VALIDATION[
    "summary_persisted"
] = SUMMARY_PATH.is_file()


print("\nVALIDATION RESULTS")
print("-" * 100)

for key, value in VALIDATION.items():

    print(
        f"{key:30s} : "
        f"{'PASSED' if value else 'FAILED'}"
    )


if not all(
    VALIDATION.values()
):

    raise RuntimeError(
        "Notebook 08 validation failed."
    )


print(
    "\nALL NOTEBOOK 08 VALIDATIONS PASSED"
)

In [ ]:
# ============================================================
# NOTEBOOK 08.15 — DRIVE PERSISTENCE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.15")
print("DRIVE PERSISTENCE VALIDATION")
print("=" * 100)


PERSISTED_FILES = [
    CONFIDENCE_PATH,
    DECISION_PATH,
    RECOVERY_PATH,
    STABILITY_PATH,
    SUMMARY_PATH
]


for path in PERSISTED_FILES:

    if not path.is_file():

        raise FileNotFoundError(
            f"Required Notebook 08 artifact missing:\n{path}"
        )


print(
    f"Validated files: {len(PERSISTED_FILES)}"
)


for path in PERSISTED_FILES:

    print(
        f"FOUND : {path}"
    )


print(
    "\nDRIVE PERSISTENCE VALIDATION PASSED"
)

In [ ]:
# ============================================================
# NOTEBOOK 08.16 — MANIFEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08.16")
print("CREATING NOTEBOOK 08 MANIFEST")
print("=" * 100)


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
            f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


MANIFEST = {

    "notebook":
        "08_Confidence_and_Abstention",

    "project":
        "AIR-LLM",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "configuration": {

        "datasets":
            DATASETS,

        "confidence_threshold":
            CONFIDENCE_THRESHOLD,

        "utility_margin_threshold":
            UTILITY_MARGIN_THRESHOLD,

        "stability_tolerance":
            STABILITY_TOLERANCE,

        "confidence_weights": {

            "llm":
                W_LLM,

            "empirical_margin":
                W_MARGIN,

            "stability":
                W_STABILITY

        },

        "master_seed":
            MASTER_SEED
    },

    "results": {

        "features":
            int(
                len(CONFIDENCE_DF)
            ),

        "accepted":
            int(
                (
                    CONFIDENCE_DF[
                        "decision"
                    ]
                    == "ACCEPT"
                ).sum()
            ),

        "abstained":
            int(
                (
                    CONFIDENCE_DF[
                        "decision"
                    ]
                    == "ABSTAIN"
                ).sum()
            ),

        "mean_confidence":
            float(
                CONFIDENCE_DF[
                    "final_confidence"
                ].mean()
            )
    },

    "artifacts": {}

}


for path in PERSISTED_FILES:

    MANIFEST[
        "artifacts"
    ][
        str(
            path.relative_to(
                PROJECT_ROOT
            )
        )
    ] = {

        "exists":
            path.is_file(),

        "size_bytes":
            path.stat().st_size,

        "sha256":
            sha256_file(path)

    }


MANIFEST_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "notebook_08_manifest.json"
)

MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MANIFEST,
        f,
        indent=2
    )


print(
    f"Manifest saved:\n{MANIFEST_PATH}"
)

In [ ]:
# ============================================================
# NOTEBOOK 08.17 — FINAL NOTEBOOK STATUS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 08 COMPLETE")
print("=" * 100)


accepted = int(
    (
        CONFIDENCE_DF[
            "decision"
        ]
        == "ACCEPT"
    ).sum()
)


abstained = int(
    (
        CONFIDENCE_DF[
            "decision"
        ]
        == "ABSTAIN"
    ).sum()
)


total = len(
    CONFIDENCE_DF
)


abstention_rate = (
    abstained
    /
    max(
        total,
        1
    )
)


mean_confidence = float(
    CONFIDENCE_DF[
        "final_confidence"
    ].mean()
)


print("\nCONFIDENCE RESULTS")
print("-" * 100)

print(
    f"Features evaluated     : {total}"
)

print(
    f"Mean confidence        : "
    f"{mean_confidence:.4f}"
)

print(
    f"Accepted decisions     : {accepted}"
)

print(
    f"Abstained decisions    : {abstained}"
)

print(
    f"Abstention rate        : "
    f"{abstention_rate:.4f}"
)


print("\nVALIDATION")
print("-" * 100)

print(
    "Confidence range       : PASSED"
)

print(
    "Decision consistency   : PASSED"
)

print(
    "Persistence            : PASSED"
)


print("\nOUTPUTS")
print("-" * 100)

print(
    f"Confidence results     : {CONFIDENCE_PATH}"
)

print(
    f"Abstention decisions   : {DECISION_PATH}"
)

print(
    f"Recovery candidates    : {RECOVERY_PATH}"
)

print(
    f"Stability results      : {STABILITY_PATH}"
)

print(
    f"Summary                : {SUMMARY_PATH}"
)

print(
    f"Manifest               : {MANIFEST_PATH}"
)


print("\n" + "=" * 100)
print(
    "ALL NOTEBOOK 08 VALIDATIONS PASSED"
)
print(
    "AIR-LLM CONFIDENCE AND ABSTENTION ANALYSIS IS COMPLETE"
)
print(
    "NOTEBOOK 09 — EXPLAINABILITY AND FINAL EXPERIMENT ANALYSIS MAY BEGIN"
)
print("=" * 100)